# Unsupervised aptamer generation using RaptGen
Step-by-step guide for using RaptGen for unsupervised aptamer generation.

## Overview
This notebook showcases the RaptGen algorithm, a generative deep learning method that learns a latent space of aptamers from sequences alone, with no labels and no protein. A convolutional encoder maps each aptamer to a point in a low-dimensional latent space, and a decoder turns that point into a profile HMM, a probabilistic model of the sequence motif, from which new motif sequences are generated. An overview of the classes and helper functions used in this notebook:


* MoleculeLoader: lazy container for aptamer data; cells may hold in-memory sequences or file paths, materialized to a DataFrame. In this notebook, the aptamer sequences are wrapped in a MoleculeLoader and its data DataFrame is passed to the generator as X.

* RaptgenGenerator: the end-to-end algorithm. It takes raw aptamer sequences, handles the encoding into integer indices, and exposes fit, transform and inverse_transform.
* RaptGenModel: the underlying estimator on integer-encoded sequences. It trains the variational autoencoder (VAE), embeds sequences, and decodes latent points back into motif sequences. RaptgenGenerator delegates to it.
* ProfileHMMSampler: the layer that turns the decoder's HMM into a sequence, either the most probable one or a random sample


## Data preparation
To train the AptaNetMLP the notebook uses:

### X:

5 random aptamer sequences of length>30 (to satisfy the default lambda value of 30 set in the PSeAAC algorithm).
Amino acid sequences extracted from the 1GNH protein molecule.
The aptamer sequences and the amino acid sequences form tuples (aptamer_sequence, protein_sequence) to form X.


In [ ]:
from pyaptamer.raptgen import RaptgenGenerator
import pandas as pd
from pyaptamer.data import MoleculeLoader
import torch
from pyaptamer.raptgen.layers import ProfileHMMSampler 


In [ ]:

aptamer_sequence = [
    "GGGAGGACGAAGACGACUCGAGACAGGCUAGGGAGGGA",
    "AAGCGUCGGAUCUACACGUGCGAUAGCUCAGUACGCGGU",
    "CGGUAUCGAGUACAGGAGUCCGACGGAUAGUCCGGAGC",
    "UAGCUAGCGAACUAGGCGUAGCUUCGAGUAGCUACGGAA",
    "GCUAGGACGAUCGCACGUGACCGUCAGUAGCGUAGGAGA",
]

# RaptGen uses DNA letters and needs equal-length sequences
seq_len = min(len(s) for s in aptamer_sequence)
unique = [s.replace("U", "T")[:seq_len] for s in aptamer_sequence]
sequences = unique * 10  # repeat to enlarge the dataset

X = MoleculeLoader(data=pd.DataFrame(sequences, columns=["aptamer"]))

In [3]:
print(X.data.head())

                                  aptamer
0  GGGAGGACGAAGACGACTCGAGACAGGCTAGGGAGGGA
1  AAGCGTCGGATCTACACGTGCGATAGCTCAGTACGCGG
2  CGGTATCGAGTACAGGAGTCCGACGGATAGTCCGGAGC
3  TAGCTAGCGAACTAGGCGTAGCTTCGAGTAGCTACGGA
4  GCTAGGACGATCGCACGTGACCGTCAGTAGCGTAGGAG


# Building the generator

RaptGen has no dataset balancing step, and no feature selection either, because it learns directly from the raw sequences. Training a RaptgenGenerator involves these steps:

1. Convert the input (a list of sequences, a DataFrame or a MoleculeLoader) into a list of aptamer sequences, using the aptamer column.
All sequences must have the same length and contain only A, C, G and T.
2. Encode each sequence as integer indices (one per base) with seq_to_indices.
3. Train the VAE inside RaptGenModel (a convolutional encoder and a profile-HMM decoder) with the Adam optimizer, annealing the weight of the KL term (beta annealing) and early stopping.
Use the fitted model to embed sequences into the latent space (transform), and to decode latent points into motif sequences (inverse_transform).

## Different workflows

In this notebook we cover the following workflows. The first three use the small dummy dataset and the last one uses real aptamer sequences:

1. Fit and embed. Train the model and map the sequences to latent-space points.
2. Generate the most likely motifs. Decode latent points with most_likely=True: the decoder enumerates candidate sequences and returns the one with the highest probability under the HMM.
3. Generate sampled motifs. Decode with most_likely=False to draw random sequences from the HMM, giving more diverse results.


# Workflow 1
Simple usage of the pipeline for random generation

In [4]:
pipeline=RaptgenGenerator(motif_len=5)

In [5]:
pipeline.fit(X.data)

,motif_len,5
,aptamer_col,'aptamer'


In [ ]:
# 1. Embed the training sequences into latent space
Z = pipeline.transform(X.data)
print(Z.shape)
# 2. Generate new sequences by sampling from the latent space
pipeline.inverse_transform(Z[:3])
# 3. Generate new sequences by sampling from the latent space, using the random sampling approach (most_likely=False)
pipeline.inverse_transform(Z[:3], most_likely=False)

(50, 10)


# Workflow 2
Simple usage of the pipeline for most_likely approach for sampling

In [9]:
# 1. Embed the training sequences into latent space
Z = pipeline.transform(X.data)
print(Z.shape)
# 2. Generate new sequences by sampling from the latent space
pipeline.inverse_transform(Z[:3])
# 3. Generate new sequences by sampling from the latent space, using the random sampling approach (most_likely=False)
pipeline.inverse_transform(Z[:3], most_likely=True)

(50, 10)


['GCGGA', 'CCGGA', 'CCGGAGG']

# Workflow 3
Using a real data file

In [ ]:

df = pd.read_csv("../pyaptamer/datasets/data/train_li2014.csv")
print(df.columns.tolist())
df.head()

['aptamer', 'protein', 'label']


,aptamer,protein,label
0,UCGGAGGUGGUUCAGCUCUGCAUCGACAGUUGGC,MDSTNYISKLFEYAQRQGQISDIKFEEVGTDGPDHLKTFTLRVVIK...,positive
1,CGGGGTGTTGTCCTGTGCTCTCCGGAGAGCACAGGACAACACCCCG,MDELFPLIFPAEPAQASGPYVEIIEQPKQRGMRFRYKCEGRSAGSI...,positive
2,GCTGCAGTTGCACTGAATTCGCCTCTCGCCTCCGTACACTTAGTCG...,MDEVGAQVAAPMFIHQSLGRKRDLYYPMSNRLVQSQPQRRDEWNSK...,positive
3,TATTTGCCCTTGCAGGCCGCAGGAGTGCTAGCAGT,PISPIDTVPVKLKPGMDGPKVKQWPLTEEKIKALTEICNEMEKEGK...,positive
4,UCUCUGGGCUCUUAGGAGAACGGAUAGGAGUGUGCUCGCU,LQENFLPQPRQQHHGTLVLHYRPHREEAGMQHPCLWPGSSHCSDDS...,positive


In [14]:
seqs = (
    df.loc[df["label"] == "positive", "aptamer"]
    .str.replace("U", "T")
    .drop_duplicates()
)

seqs = seqs[seqs.str.fullmatch(r"[ACGT]+")]

common_len = seqs.str.len().value_counts().idxmax()
seqs = seqs[seqs.str.len() == common_len]
print(f"{len(seqs)} unique sequences of length {common_len}")

seqs = seqs.sample(n=min(200, len(seqs)), random_state=0)
X_real = MoleculeLoader(data=pd.DataFrame({"aptamer": seqs.tolist()}))

77 unique sequences of length 40


In [15]:
pipeline_real = RaptgenGenerator(motif_len=10)
pipeline_real.fit(X_real.data)
Z_real = pipeline_real.transform(X_real.data)
pipeline_real.inverse_transform(Z_real[:3])

KeyboardInterrupt: 

# Sequence scoring
Each latent point defines a profile HMM, so any motif can be scored by how probable it is under that HMM. A higher (less negative) log-probability means a better fit. calc_seq_proba computes this score with the forward algorithm. 

In [ ]:

model = pipeline.model_.model_   
z = torch.tensor(Z[:1], dtype=torch.float32)

with torch.no_grad():
    a, e = model.decoder(z)      

with torch.no_grad():
    a, e = model.decoder(z)

sampler = ProfileHMMSampler(
    a[0].detach().cpu().numpy(),
    e[0].detach().cpu().numpy(),
    proba_is_log=True,
)
sampler.calc_seq_proba("ACGTAC")   

tensor(-12.8244)